# PCPT Data Processing and Integration
This notebook automates the workflow for cleaning, converting, and merging PCPT test data with Post-DC parameters.
It demonstrates file handling, data normalization, and diagnostic logging for geotechnical analysis.


## Step 1: Clean up old files
We remove previously generated Excel and CSV files to ensure a fresh run.


In [ ]:
import glob
import os

# Patterns for both Excel and CSV files
file_patterns = ["PCPT-*.xlsx", "PCPT-*.csv"]

for pattern in file_patterns:
    files = glob.glob(pattern)
    for file in files:
        try:
            os.remove(file)
            print(f"Deleted {file}")
        except Exception as e:
            print(f"Could not delete {file}: {e}")


## Step 2: Convert CSV to Excel
We normalize headers and save each PCPT dataset into Excel format for consistency.


In [ ]:
import pandas as pd
import glob
import os

# Collect all PCPT CSV files
pcpt_files = glob.glob("PCPT-*.csv")

# Loop through each file and convert to Excel
for file in pcpt_files:
    # Skip metadata lines, use the 4th line as header
    df = pd.read_csv(file, skiprows=3, header=0)

    # Normalize column names
    df.columns = df.columns.str.strip()

    # Rename columns for consistency
    df.rename(columns={
        "Depth [m]": "Depth",
        "qc [MPa]": "qc",
        "fs [MPa]": "fs",
        "u2 [MPa]": "u2"
    }, inplace=True)

    # Create output filename
    out_file = os.path.splitext(file)[0] + ".xlsx"

    # Save to Excel
    df.to_excel(out_file, index=False)

    print(f"Converted {file} -> {out_file} with proper headers")


## Step 3: Merge with Post-DC Parameters
We align PCPT qc values with Post-DC parameters, log unmatched values, and generate a combined master file.


In [ ]:
import pandas as pd
import glob
import os

# === Load Pre-DC parameters ===
pre_dc = pd.read_excel("/content/Post-DC parameters.xlsx", sheet_name=0)
pre_dc.columns = pre_dc.columns.str.strip().str.lower()

# Round qc values to 3 decimals for consistency
pre_dc["qc"] = pre_dc["qc"].round(3)

print("Pre-DC columns:", pre_dc.columns.tolist())
print("Pre-DC sample:\n", pre_dc.head())

# === Collect PCPT Excel files ===
pcpt_files = glob.glob("PCPT-*.xlsx")
print("Found PCPT Excel files:", pcpt_files)

summary_data = []

with pd.ExcelWriter("Master_Combined.xlsx") as writer:
    for file in pcpt_files:
        print("\nProcessing file:", file)

        # Read Excel (no headers in file, so assign manually)
        pcpt = pd.read_excel(file, header=None)
        pcpt.columns = ["depth", "qc", "fs", "u2"]

        print("Assigned columns:", pcpt.columns.tolist())
        print("First 5 rows:\n", pcpt.head())

        # Round qc values
        pcpt["qc"] = pcpt["qc"].round(3)

        # Filter depths
        subset = pcpt[(pcpt["depth"] >= 1.0) & (pcpt["depth"] <= 5.99)][["depth","qc"]]
        print("Subset rows:", len(subset))
        print("Subset sample:\n", subset.head())

        # === Exact merge (like XLOOKUP default) ===
        merged = subset.merge(pre_dc, on="qc", how="left")

        # Log unmatched qc values
        unmatched_qc = subset[~subset["qc"].isin(pre_dc["qc"])]
        print("Unmatched qc values (first 10):", unmatched_qc["qc"].head(10).tolist())

        print("Merged rows:", len(merged))
        print("Merged sample:\n", merged.head())

        # Deduplicate
        merged = merged.drop_duplicates(subset=["depth","qc"], keep="first")

        # Restrict columns
        final_cols = ["depth","qc","ic","dr","ef","fa","e","f"]
        merged = merged[[c for c in final_cols if c in merged.columns]]

        # Diagnostics
        unmatched = merged[["ic","dr","ef","fa","e","f"]].isna().sum().sum()
        summary_data.append({
            "File": os.path.basename(file),
            "RowsExtracted": len(merged),
            "UnmatchedValues": unmatched
        })

        # Write sheet
        sheet_name = os.path.basename(file).replace(".xlsx", "")[:31]
        merged.to_excel(writer, sheet_name=sheet_name, index=False)

    # Add summary sheet
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
